# Colour scale exploration

Five topics across six expanding date ranges (2020 only to 2020-present).

- **Spiral grid**: default cutoff (2 x P75) so you can see how the colour scale behaves per topic.
- **Cutoff method comparison**: line graphs with five candidate cutoff rules overlaid.
  Click a method in the legend to toggle it across all panels.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from seasonal_spirals import fetch_pageviews
from seasonal_spirals._colourmap import WIKISPIRAL_PLOTLY, HybridNorm, auto_cutoff
from seasonal_spirals._geometry import (
    N_WEEKS,
    spiral_year,
    spiral_year_start,
    trim_to_max_years,
    tile_geometry,
    month_label_positions,
)

## Fetch data

In [ ]:
START = "2020-01-01"
TODAY = pd.Timestamp.today().strftime("%Y-%m-%d")

ARTICLES = [
    "Influenza",
    "Kayaking",
    "Rebecca Black",
    "Taco",
    "Electric Brae",
]

raw = {}
for article in ARTICLES:
    print(f"Fetching {article} ...")
    raw[article] = fetch_pageviews(article, start=START, end=TODAY)
    s = raw[article]
    print(f"  {len(s)} days  ({s.index[0].date()} to {s.index[-1].date()})  max={s.max():,.0f}")

## Setup

In [ ]:
periods = [
    ("2020",          START, "2020-12-31"),
    ("2020-2021",     START, "2021-12-31"),
    ("2020-2022",     START, "2022-12-31"),
    ("2020-2023",     START, "2023-12-31"),
    ("2020-2024",     START, "2024-12-31"),
    ("2020-present", START, TODAY),
]

topics = [(a, raw[a]) for a in ARTICLES]

N_ROWS = len(periods)
N_COLS = len(topics)


def _mad_cutoff(v):
    med = np.median(v)
    mad = np.median(np.abs(v - med))
    return med + 3.0 * 1.4826 * mad


def _iqr_cutoff(v):
    q1, q3 = np.percentile(v, 25), np.percentile(v, 75)
    return q3 + 1.5 * (q3 - q1)


def _lognormal_cutoff(v):
    log_v = np.log(np.maximum(v, 1.0))
    return float(np.exp(np.mean(log_v) + 2.0 * np.std(log_v)))


# (label, function, colour, dash)
CUTOFF_METHODS = [
    ("2xP75 (current)",    lambda v: 2.0 * np.percentile(v, 75), "#e41a1c", "solid"),
    ("2xP50 (median)",     lambda v: 2.0 * np.percentile(v, 50), "#984ea3", "dash"),
    ("MAD (Hampel k=3)",   _mad_cutoff,                           "#377eb8", "dot"),
    ("Tukey IQR (Q3+1.5)", _iqr_cutoff,                          "#4daf4a", "dashdot"),
    ("Log-normal fit",     _lognormal_cutoff,                     "#ff7f00", "longdash"),
]


def _polar_key(row, col):
    idx = (row - 1) * N_COLS + col
    return "polar" if idx == 1 else f"polar{idx}"


def add_spiral_panel(
    fig,
    data: pd.Series,
    row: int,
    col: int,
    start_month: int = 1,
    inner_radius: float = 0.1,
    ring_width: float = 1.0,
    week_gap: float = 0.06,
    year_gap: float = 0.15,
    max_years: int = 9,
    cutoff_fn=None,
) -> float:
    """Add a WikiSpiral panel. Returns the cutoff value used.

    cutoff_fn: optional callable(vals) -> float. Defaults to auto_cutoff (2xP75).
    Result is clamped to [vmin+5%, vmax-5%] so both colour segments always exist.
    """
    data = data.dropna().sort_index()
    if getattr(data.index, "tz", None) is not None:
        data = data.copy()
        data.index = data.index.tz_localize(None)
    data = trim_to_max_years(data, start_month, max_years)

    vals = data.values.astype(float)
    vmin_ = float(np.nanmin(vals))
    vmax_ = float(np.nanmax(vals))

    if cutoff_fn is not None:
        raw_cutoff = float(cutoff_fn(vals))
        # Same clamping auto_cutoff applies - ensures both segments have extent
        lo = vmin_ + 0.05 * (vmax_ - vmin_)
        hi = vmax_ - 0.05 * (vmax_ - vmin_)
        cutoff = float(np.clip(raw_cutoff, lo, hi))
    else:
        cutoff = auto_cutoff(vals, vmin_, vmax_)

    hybrid_norm = HybridNorm(vmin_, vmax_, cutoff)

    week_increment = (ring_width + year_gap) / N_WEEKS
    day_band = ring_width / 7.0

    spiral_years = sorted(set(spiral_year(dt, start_month) for dt in data.index))
    min_year = spiral_years[0]

    r_vals, base_vals, theta_vals, width_vals = [], [], [], []
    colour_vals, hover_texts = [], []

    for dt, value in data.items():
        sy = spiral_year(dt, start_month)
        ys = spiral_year_start(sy, start_month)
        year_idx = sy - min_year
        day_offset = (dt.normalize() - ys).days
        weekday = dt.weekday()

        arc_start_rad, arc_width_rad, r_inner, r_outer = tile_geometry(
            day_offset, year_idx, weekday,
            inner_radius, ring_width, week_gap, year_gap,
        )
        theta_centre = np.degrees(arc_start_rad) + np.degrees(arc_width_rad) / 2.0

        r_vals.append(r_outer - r_inner)
        base_vals.append(r_inner)
        theta_vals.append(theta_centre)
        width_vals.append(np.degrees(arc_width_rad))
        colour_vals.append(float(value))
        hover_texts.append(
            f"<b>{dt.strftime('%a %d %b %Y')}</b><br>"
            f"Views: {float(value):,.0f}<br>"
            f"Cutoff: {cutoff:,.0f}"
        )

    colour_norm = list(hybrid_norm(np.array(colour_vals, dtype=float)))

    fig.add_trace(
        go.Barpolar(
            r=r_vals, base=base_vals,
            theta=theta_vals, width=width_vals,
            marker=dict(
                color=colour_norm,
                colorscale=WIKISPIRAL_PLOTLY,
                cmin=0.0, cmax=1.0,
                showscale=False,
                line=dict(width=0),
            ),
            hovertext=hover_texts,
            hoverinfo="text",
            showlegend=False,
        ),
        row=row, col=col,
    )

    yr_r = [
        inner_radius + (i * N_WEEKS + N_WEEKS // 2) * week_increment + 3 * day_band
        for i in range(len(spiral_years))
    ]
    fig.add_trace(
        go.Scatterpolar(
            r=yr_r, theta=[4.0] * len(yr_r),
            mode="text", text=[f"<b>{sy}</b>" for sy in spiral_years],
            textfont=dict(size=6, color="#999999"),
            textposition="middle right",
            hoverinfo="skip", showlegend=False,
        ),
        row=row, col=col,
    )

    last_dt = data.index[-1]
    last_sy = spiral_year(last_dt, start_month)
    last_ys = spiral_year_start(last_sy, start_month)
    last_day_offset = (last_dt.normalize() - last_ys).days
    last_week = min(last_day_offset // 7, N_WEEKS - 1)
    last_year_idx = last_sy - min_year

    label_tuples = month_label_positions(
        last_sy, last_ys, start_month,
        inner_radius, ring_width, year_gap,
        last_year_idx, last_week,
    )
    month_r     = [r             for _, _, _, r in label_tuples]
    month_theta = [np.degrees(a) for a, _, _, _ in label_tuples]
    month_text  = [abbrev        for _, abbrev, _, _ in label_tuples]

    fig.add_trace(
        go.Scatterpolar(
            r=month_r, theta=month_theta,
            mode="text", text=month_text,
            textfont=dict(size=6, color="#555555"),
            hoverinfo="skip", showlegend=False,
        ),
        row=row, col=col,
    )

    r_max = max(month_r) + 0.3
    fig.update_layout({
        _polar_key(row, col): dict(
            angularaxis=dict(
                direction="clockwise", rotation=90,
                showticklabels=False, showgrid=False, showline=False,
            ),
            radialaxis=dict(visible=False, range=[0, r_max]),
            bgcolor="white",
        )
    })

    return cutoff

## Spiral grid (default cutoff: 2 x P75)

In [ ]:
spiral_titles = [
    f"{topic} - {label}"
    for label, _, _ in periods
    for topic, _ in topics
]

spiral_fig = make_subplots(
    rows=N_ROWS, cols=N_COLS,
    specs=[[{"type": "polar"}] * N_COLS for _ in range(N_ROWS)],
    subplot_titles=spiral_titles,
    horizontal_spacing=0.02,
    vertical_spacing=0.04,
)

cutoffs = {}
for row, (period_label, start, end) in enumerate(periods, start=1):
    for col, (topic_name, data_all) in enumerate(topics, start=1):
        cutoffs[(row, col)] = add_spiral_panel(
            spiral_fig, data_all[start:end], row=row, col=col
        )

spiral_fig.update_layout(
    title=dict(text="WikiSpiral colour scale - expanding date ranges (default cutoff: 2 x P75)", font=dict(size=13)),
    height=N_ROWS * 380,
    width=N_COLS * 260,
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(t=60, b=20, l=20, r=20),
)
spiral_fig.show()

## Cutoff method comparison

Five methods from the outlier detection literature, all plotted on the same time series.
Click a method in the legend to toggle it across all 30 panels.

| Method | Formula | Properties |
|---|---|---|
| 2xP75 (current) | 2 * 75th percentile | Baseline; tends high when P75 is already in the signal |
| 2xP50 (median) | 2 * median | Lower anchor; more robust but still percentile-based |
| MAD (Hampel k=3) | median + 3 * 1.4826 * MAD | Gold standard for robust outlier detection; 50% breakdown point |
| Tukey IQR (Q3+1.5) | Q3 + 1.5 * (Q3-Q1) | Standard boxplot upper whisker; well-known, distribution-free |
| Log-normal fit | exp(mean(log x) + 2*std(log x)) | Appropriate for count/rate data; fits log-normal, uses 97.7th pct of that fit |

In [ ]:
line_titles = [
    f"{topic} - {label}"
    for label, _, _ in periods
    for topic, _ in topics
]

line_fig = make_subplots(
    rows=N_ROWS, cols=N_COLS,
    subplot_titles=line_titles,
    horizontal_spacing=0.06,
    vertical_spacing=0.06,
)

for row, (period_label, start, end) in enumerate(periods, start=1):
    for col, (topic_name, data_all) in enumerate(topics, start=1):
        data = data_all[start:end].dropna().sort_index()
        vals = data.values.astype(float)
        x0, x1 = data.index[0], data.index[-1]
        is_first = (row == 1 and col == 1)

        # Time series
        line_fig.add_trace(
            go.Scatter(
                x=data.index, y=vals,
                mode="lines",
                line=dict(color="#cccccc", width=0.8),
                hovertemplate="%{x|%d %b %Y}<br>Views: %{y:,.0f}<extra></extra>",
                showlegend=False,
            ),
            row=row, col=col,
        )

        # One horizontal line per cutoff method
        for method_name, method_fn, colour, dash in CUTOFF_METHODS:
            cutoff_val = method_fn(vals)
            line_fig.add_trace(
                go.Scatter(
                    x=[x0, x1],
                    y=[cutoff_val, cutoff_val],
                    mode="lines",
                    name=method_name,
                    legendgroup=method_name,
                    showlegend=is_first,
                    line=dict(color=colour, width=2, dash=dash),
                    hovertemplate=f"{method_name}: %{{y:,.0f}}<extra></extra>",
                ),
                row=row, col=col,
            )

line_fig.update_layout(
    title=dict(text="Cutoff method comparison - daily views with five candidate rules", font=dict(size=13)),
    height=N_ROWS * 160,
    width=N_COLS * 260,
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        x=0.5, xanchor="center",
        y=-0.04, yanchor="top",
        font=dict(size=10),
    ),
    margin=dict(t=50, b=70, l=40, r=20),
)
line_fig.update_xaxes(showgrid=False, tickformat="%b %Y", tickfont=dict(size=7))
line_fig.update_yaxes(showgrid=True, gridcolor="#eeeeee", tickformat=",", tickfont=dict(size=7))
line_fig.show()

## Spiral grid - Tukey IQR cutoff (Q3 + 1.5 * IQR)

In [ ]:
iqr_spiral_titles = [
    f"{topic} - {label}"
    for label, _, _ in periods
    for topic, _ in topics
]

iqr_fig = make_subplots(
    rows=N_ROWS, cols=N_COLS,
    specs=[[{"type": "polar"}] * N_COLS for _ in range(N_ROWS)],
    subplot_titles=iqr_spiral_titles,
    horizontal_spacing=0.02,
    vertical_spacing=0.04,
)

for row, (period_label, start, end) in enumerate(periods, start=1):
    for col, (topic_name, data_all) in enumerate(topics, start=1):
        add_spiral_panel(
            iqr_fig, data_all[start:end], row=row, col=col,
            cutoff_fn=_iqr_cutoff,
        )

iqr_fig.update_layout(
    title=dict(text="WikiSpiral - Tukey IQR cutoff (Q3 + 1.5 * IQR)", font=dict(size=13)),
    height=N_ROWS * 380,
    width=N_COLS * 260,
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(t=60, b=20, l=20, r=20),
)
iqr_fig.show()